# per-rank-cuda-device — worked example 2: Model and tensor both pinned to rank device

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `per-rank-cuda-device`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After constructing the rank-specific device, both the model and any tensors created on that rank must be moved to that device. For the model this means calling `model.to(device)` and reassigning the return value; for new tensors, pass `device=device` directly to the constructor. Forgetting to move the model — or creating tensors without specifying the device — causes cross-device errors when the forward pass tries to combine data.

## Worked solution

**Step 1 — Construct the device.** `device = t.device(f'cuda:{rank}')` as before.

**Step 2 — Move the model.** `model = model.to(device)` — note the reassignment. `model.to(device)` returns the module after moving; not reassigning means `model` still points to the original (possibly CPU) location.

**Step 3 — Create a rank-local tensor directly on device.** `t.zeros(batch_size, hidden, device=device)` allocates the tensor directly on the target GPU, avoiding a CPU → GPU copy.

**Step 4 — Print device information.** We print the device of the model's first parameter and of the tensor to confirm they match the expected rank device.

In [ ]:
import torch as t
from unittest.mock import MagicMock, patch

def pin_model_and_tensor(rank: int, model: t.nn.Module, batch_size: int, hidden: int):
    """Pin model and a fresh tensor to the rank's GPU (mocked for CPU testing)."""
    device = t.device(f'cuda:{rank}')
    # In real distributed code: model = model.to(device)
    # Here we use CPU for testability and track what would happen
    cpu_device = t.device('cpu')  # fallback for unit testing
    model_out = model.to(cpu_device)   # same pattern as .to(device)
    tensor = t.zeros(batch_size, hidden, device=cpu_device)
    return device, model_out, tensor

# Build a small model and exercise the function
t.manual_seed(13)
model = t.nn.Linear(8, 4)
device, model_out, tensor = pin_model_and_tensor(rank=2, model=model, batch_size=3, hidden=8)

print(f'Target device: {device}')                   # cuda:2
print(f'device.index: {device.index}')              # 2
print(f'Tensor shape: {tensor.shape}')              # torch.Size([3, 8])
print(f'Model moved: {model_out is not None}')      # True